# Modellierungsseminar Sommer 2026
## Cycle Planning for workforce scheduling

In [2]:
try:
    !pip install gurobipy
except:
    %pip install gurobipy

In [32]:
!git clone -b feature/shift-objects https://github.com/poprjaduhhaa/Modellierungsseminar-Firestation.git
%cd /content/Modellierungsseminar-Firestation/coding

fatal: destination path 'Modellierungsseminar-Firestation' already exists and is not an empty directory.
/content/Modellierungsseminar-Firestation/coding


In [33]:
# import required packages
import gurobipy as gp
from gurobipy import GRB
import pandas as pd
from dataclasses import dataclass
import datetime as dt
import src.Shift as Shift # tailor-made data type for shift definitions

### inputs and parameters

In [34]:

# basic inputs and parameters

Weekdays = range(1,8)  # results in 1,...,7 => let 1 be Monday and 7 be Sunday
Shifts = ["frueh", "spaet", "nacht", "frei"] # including free shifts
WorkShifts = ["frueh", "spaet", "nacht"] # excluding free shifts

# improvements outstanding:
    # use files for parameter input
        # shift definitions
        # available staff
        # user objectives: weighted priorities


# more relevant for more complex models going forward
MAX_CYCLE_WEEKS = int(52/4)  # in future this shall be user's input => too long snakes do not help to ensure "fair" distribution of shifts
DICT_WEEKDAYS = {'Mon':1,'Tue':2,'Wed':3,'Thu':4,'Fri':5,'Sat':6,'Sun':7,
                 'Monday':1,'Tuesday':2,'Wednesday':3,'Thursday':4,'Friday':5,'Saturday':6,'Sunday':7,
                 'Mo':1,'Tu':2,'We':3,'Th':4,'Fr':5,'Sa':6,'Su':7,
                 '1':1,'2':2,'3':3,'4':4,'5':5,'6':6,'7':7

}


In [35]:
# read input data
# shift set

def readShiftSet(filename: str, mySep: str=";") -> pd.DataFrame:
    input_data = pd.read_csv(filename, sep=mySep, dtype=str) # import all values as string as first step
    return input_data
    # potentially add data cleaning steps

# added objects to read whole csv file
def build_shift_objects(df: pd.DataFrame) -> list:
    shift_objects = []
    for _, row in df.iterrows():
        s = Shift.Shift(
            shift_id=row['shift_ID'],
            description=row['shift_details'],
            weekdays=[d.strip() for d in row['shift_weekdays'].split(',')],
            start=dt.time(*map(int, row['shift_start_time'].split(':'))),
            end=dt.time(*map(int, row['shift_end_time'].replace('24','0').split(':'))),
            required_staff=0 if row['shift_required_staff'] == 'none' else int(row['shift_required_staff']),
            shift_class=int(row['shift_class']),
            shift_work_time_assignment=str(row['shift_work_time_assignment']),
            is_work_shift=bool(row['isWorkShift']),
            required_qualification=row['[shift_required_qualification]']
        )
        shift_objects.append(s)
    return shift_objects

folderpath = "input/"
filename = "input_ShiftDataSet.csv"

data_shiftSet = readShiftSet(folderpath + filename)
data_shiftSet["isWorkShift"] = data_shiftSet["isWorkShift"].astype(int).astype(bool)
#print(data_shiftSet)

shift_objects = build_shift_objects(data_shiftSet)
for s in shift_objects:
    print(s)

Shifts = [s.shift_id for s in shift_objects]
WorkShifts = [s.shift_id for s in shift_objects if s.is_work_shift] #object oriented solution

#Shifts = list(data_shiftSet["shift_ID"]) # including free shifts
#WorkShifts = list(data_shiftSet[data_shiftSet["isWorkShift"]]["shift_ID"]) # excluding free shifts


#print(Shifts)
#print(data_shiftSet["shift_weekdays"])



Shift(shift_id='000001EARLY0', description='early day shift', weekdays=['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun'], start=datetime.time(6, 0), end=datetime.time(14, 0), required_staff=5, shift_class=1, shift_work_time_assignment='nan', is_work_shift=True, required_qualification='none')
Shift(shift_id='000002LATE00', description='late day and afternoon shift', weekdays=['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun'], start=datetime.time(14, 0), end=datetime.time(22, 0), required_staff=4, shift_class=2, shift_work_time_assignment='nan', is_work_shift=True, required_qualification='none')
Shift(shift_id='000003NIGHT0', description='night shift', weekdays=['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun'], start=datetime.time(22, 0), end=datetime.time(6, 0), required_staff=3, shift_class=5, shift_work_time_assignment='nan', is_work_shift=True, required_qualification='none')
Shift(shift_id='000000FREEDY', description='free day', weekdays=['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun

### modelling

In [36]:
# modelling

m = gp.Model("SnakeBuilding_simple")

# variables:
# x[s, d, sh] = 1, when snake s is working in shift sh on day d
x = m.addVars(MAX_CYCLE_WEEKS, Weekdays, Shifts, vtype=GRB.BINARY, name="x")

# active[s] = 1, when snake s is used
active = m.addVars(MAX_CYCLE_WEEKS, vtype=GRB.BINARY, name="active") # all other snakes are used as placeholders but not necessarily get activated


### conditions:

#### condition c01
_(idea is to use a unique ID for each condition for better reference)_

each shift has to be covered on each day

$$\sum_{s=1}^{n}{x_{s,d,w}} >= 1    \forall d \in D, \forall w \in W$$

$x_{s,d,w} = 1$, when snake s is working in work shift w on day d

$x: $ binary variable,
$s: $ snake number,
$d: $ weekday,
$ws: $ work shift


In [37]:
# SUBJECT TO:

# 1. each shift has to be covered on each day
for d in Weekdays:
    for ws in WorkShifts:
        m.addConstr(gp.quicksum(x[s, d, ws] for s in range(MAX_CYCLE_WEEKS)) >= 1,
                    name=f"Cover_day{d}_{ws}")


####

#### condition c02

In [38]:
# 2. each snake can have at most one shift per day
for s in range(MAX_CYCLE_WEEKS):
    for d in Weekdays:
        m.addConstr(gp.quicksum(x[s, d, sh] for sh in Shifts) == active[s],
                    name=f"OneShiftPerDay_s{s}_d{d}")


#### condition c03

In [39]:

# 3) at max 5 consecutive working days (ensure time for resting)
for s in range(MAX_CYCLE_WEEKS):
    for start in range(1, 7-5+1):
        m.addConstr(
            gp.quicksum(x[s, d, sh] for d in range(start, start + 6)
                        for sh in WorkShifts) <= 5,
            name=f"Max5Work_s{s}_start{start}"
        )



### objective

In [40]:
# set objective function: minimize number of active snakes
m.setObjective(gp.quicksum(active[s] for s in range(MAX_CYCLE_WEEKS)), GRB.MINIMIZE)

# improvements outstanding:
    # add various weighted objectives

#run optimizer
m.optimize()



Gurobi Optimizer version 13.0.2 build v13.0.2rc1 (linux64 - "Ubuntu 22.04.5 LTS")

CPU model: Intel(R) Xeon(R) CPU @ 2.20GHz, instruction set [SSE2|AVX|AVX2]
Thread count: 1 physical cores, 2 logical processors, using up to 2 threads

Optimize a model with 145 rows, 468 columns and 1534 nonzeros (Min)
Model fingerprint: 0x3cfa3f4d
Model has 13 linear objective coefficients
Variable types: 0 continuous, 468 integer (468 binary)
Coefficient statistics:
  Matrix range     [1e+00, 1e+00]
  Objective range  [1e+00, 1e+00]
  Bounds range     [1e+00, 1e+00]
  RHS range        [1e+00, 5e+00]

Found heuristic solution: objective 5.0000000
Presolve removed 30 rows and 143 columns
Presolve time: 0.01s
Presolved: 115 rows, 325 columns, 1014 nonzeros
Variable types: 0 continuous, 325 integer (325 binary)

Root relaxation: objective 4.000000e+00, 194 iterations, 0.00 seconds (0.00 work units)

    Nodes    |    Current Node    |     Objective Bounds      |     Work
 Expl Unexpl |  Obj  Depth IntInf 

### results

In [41]:
# output (raw version, to be improved for better readability)
    # improvements outstanding:
        # write results in file
        # create a shift overview per staff member

if m.status == GRB.OPTIMAL:
    print("\nminimum number of cycle weeks:", int(m.objVal))
    for s in range(MAX_CYCLE_WEEKS):
        if active[s].X == 1:
            print(f"\ncycle week {s+1}:")
            for d in Weekdays:
                for sh in Shifts:
                    if x[s, d, sh].X == 1:
                        print(f"  day {d}: {sh}")



minimum number of cycle weeks: 5

cycle week 3:
  day 1: 000001EARLY0
  day 2: 000000FREEDY
  day 3: 000004WHOLED
  day 4: 000002LATE00
  day 5: 000003NIGHT0
  day 6: 000003NIGHT0
  day 7: 000003NIGHT0

cycle week 5:
  day 1: 000000FREEDY
  day 2: 000001EARLY0
  day 3: 000002LATE00
  day 4: 000001EARLY0
  day 5: 000000FREEDY
  day 6: 000001EARLY0
  day 7: 000000FREEDY

cycle week 6:
  day 1: 000003NIGHT0
  day 2: 000002LATE00
  day 3: 000001EARLY0
  day 4: 000003NIGHT0
  day 5: 000002LATE00
  day 6: 000000FREEDY
  day 7: 000001EARLY0

cycle week 10:
  day 1: 000004WHOLED
  day 2: 000003NIGHT0
  day 3: 000003NIGHT0
  day 4: 000000FREEDY
  day 5: 000004WHOLED
  day 6: 000004WHOLED
  day 7: 000002LATE00

cycle week 12:
  day 1: 000002LATE00
  day 2: 000004WHOLED
  day 3: 000000FREEDY
  day 4: 000004WHOLED
  day 5: 000001EARLY0
  day 6: 000002LATE00
  day 7: 000004WHOLED


In [42]:
print(shift_objects[1])

Shift(shift_id='000002LATE00', description='late day and afternoon shift', weekdays=['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun'], start=datetime.time(14, 0), end=datetime.time(22, 0), required_staff=4, shift_class=2, shift_work_time_assignment='nan', is_work_shift=True, required_qualification='none')
